<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-28</br>
</div>

</br>

In [ ]:
# TODO 0: 환경변수를 로드하고, LLM, 검색기, 도구, StateGraph 구성요소를 준비해봅시다.

import os, warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()

# LLM 초기화
from langchain_upstage import ChatUpstage, UpstageEmbeddings

llm = ChatUpstage(model="solar-pro3", temperature=0)

# 정책 문서 정의
from langchain_core.documents import Document

refund_policy = """# 환불 정책 (Refund Policy)

## 환불 조건
1. 구매 후 7일 이내: 무조건 환불 가능 (환불 승인)
2. 구매 후 7일 초과: 환불 불가 (환불 거부)
3. 상품 불량 시: 30일 이내 환불 가능

## 환불 절차
- 환불 요청 접수 → 상태 확인 → 승인/거부 → 처리 완료
- 승인 시 3~5 영업일 내 환불 처리

## 주의사항
- 사용 흔적이 있는 상품은 환불 불가
- 디지털 상품은 다운로드 후 환불 불가"""

reward_policy = """# 보상 정책 (Reward Policy)

## 쿠폰 발급 조건
1. 배송 지연 시: 최대 10,000원 쿠폰 발급 가능
2. 상품 불량 시: 최대 20,000원 쿠폰 발급 가능
3. 단순 변심: 쿠폰 발급 불가

## 쿠폰 발급 한도
- 1회 최대 발급 금액: 20,000원
- 월간 누적 한도: 50,000원

## 주의사항
- 배송 완료 후 7일 이내에만 보상 가능
- 이미 쿠폰을 받은 주문에는 추가 발급 불가"""

policy_documents = [
    Document(page_content=refund_policy, metadata={"source": "환불 정책"}),
    Document(page_content=reward_policy, metadata={"source": "보상 정책"}),
]

# 청킹 → 임베딩 → 벡터스토어 → 검색기
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
split_docs = text_splitter.split_documents(policy_documents)

embeddings = UpstageEmbeddings(model="embedding-query")
vectorstore = Chroma.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# search_policy 도구 정의 (001에서 학습)
from langchain_core.tools import tool

@tool
def search_policy(query: str) -> str:
    """고객 서비스 정책을 검색합니다."""
    documents = retriever.invoke(query)
    return "\n".join([doc.page_content for doc in documents])

# 도구 바인딩 및 실행 노드 (001에서 학습)
from langgraph.prebuilt import ToolNode

tools = [search_policy]
llm_with_tools = llm.bind_tools(tools)
tool_node = ToolNode(tools)

def call_model(state):
    """LLM을 호출하여 응답 생성"""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# StateGraph 구성요소
from langgraph.graph import MessagesState, StateGraph, START, END
from langchain_core.messages import HumanMessage

print(f"✅ 환경 설정 완료 (정책 문서 {len(split_docs)}개 청크)")

</br>

# 학습 내용
>이번 장에서는 <strong>StateGraph와 ReAct 패턴(StateGraph & ReAct Pattern)</strong>에 대해 학습합니다.
>Direct 패턴과 ReAct 패턴의 차이를 이해하고, StateGraph로 Agent 그래프를 직접 학습해봅시다.

<div style="text-align:center">

</div>

</br>

# Direct 패턴 (Direct Pattern)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">한 번의 LLM 호출</mark>로 도구 사용 여부를 결정하고 바로 응답하는 단순 패턴입니다.

> 실제 업무에서 AI가 처리해야 하는 작업은 단순 Q&A를 넘어섭니다.</br></br>
> "3일 전에 산 상품 환불 가능해?"라는 질문 하나에도 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정책 검색 → 날짜 계산 → 조건 판단 → 답변 생성</mark>의 여러 단계가 필요합니다.</br></br>
> <strong>StateGraph</strong>는 이런 복잡한 워크플로우를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">노드(처리 단계)와 엣지(흐름 방향)로 구조화</mark>하여 관리할 수 있게 하고,</br></br>
> <strong>ReAct 패턴</strong>은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Reasoning(추론) + Acting(행동)</mark>의 합성어로, LLM이 생각하고 → 도구를 사용하고 → 결과를 관찰하고 → 다시 생각하는 루프를 구현합니다.</br></br>
> 이 패턴이 없으면 다단계 작업을 코드로 하드코딩해야 하지만, StateGraph + ReAct를 사용하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LLM이 스스로 판단하며 단계를 진행</mark>할 수 있습니다.</br></br>

In [ ]:
# TODO 1: 도구 호출 여부를 판단하는 분기 함수(should_continue)를 정의하세요. 마지막 메시지에 도구 호출이 있으면 "tools", 없으면 종료를 반환하세요.

In [ ]:
# TODO 2: 상태 그래프로 Direct 패턴 그래프를 구성하세요. agent→tools→종료 흐름으로 엣지를 연결하고, "환불 정책 알려줘"로 실행하여 결과를 출력하세요.

# Direct 패턴 그래프

</br>

# ReAct 패턴 (Reasoning + Acting)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">추론(Reason)과 행동(Act)을 반복</mark>하며 복잡한 질문을 단계적으로 해결합니다.

In [ ]:
# TODO 3: ReAct 패턴 그래프를 구성하세요. Direct 패턴과 동일하지만, 도구 실행 후 다시 agent로 돌아가도록 엣지를 변경하세요. "3일 전에 산 상품 환불 가능해?"로 실행하여 결과를 출력하세요.

</br>

## Direct vs ReAct 비교

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">비교 항목</th>
      <th>Direct</th>
      <th>ReAct</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">도구 호출</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">1회</mark></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">반복 가능</mark></td></tr>
    <tr><td style="text-align:center">복잡한 질문</td><td>한계 있음</td><td>단계적 해결</td></tr>
    <tr><td style="text-align:center">도구→이후</td><td>END (종료)</td><td>agent (재추론)</td></tr>
    <tr><td style="text-align:center">비용</td><td>낮음</td><td>상대적으로 높음</td></tr>
  </tbody>
</table>

</br>

## conditional_edges (조건부 분기)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">상태에 따라 다른 노드로 분기</mark>하는 핵심 메커니즘입니다.

In [ ]:
# TODO 4: 조건부 엣지를 사용하여 "agent" 노드에서 분기 함수에 따라 "tools" 또는 종료로 분기하도록 설정하세요.

# 테스트 1: 도구가 필요한 질문 → agent → tools → agent → END

# 테스트 2: 도구 없이 답변 가능한 질문 → agent → END

💡ReAct의 핵심
> 도구 실행 결과를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">다시 LLM에 피드백</mark>하여 추가 추론이 가능합니다.
> "환불 정책 검색 → 조건 확인 → 환불 처리" 같은 다단계 작업에 적합합니다.

💡ReAct 무한 루프 방지
> ReAct는 이론적으로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">무한 반복</mark>할 수 있습니다.
> `recursion_limit`을 설정하여 최대 반복 횟수를 제한해야 합니다.